# Open RAG From Scratch — Parts 5–9

Phase 1 built the whole chain: **load → split → embed → store**, then **embed the question → top-k → generate**.
That works, but it searches exactly once, using the question precisely as it was asked.

**Phase 2 changes only the query.** Rather than embedding the raw question, we ask a language model to build a
better *search input* first — several paraphrases, some sub-questions, a more general "step-back" question, or a
hypothetical answer — and then retrieve and generate exactly as before.

| Part | Technique | What changes |
|---|---|---|
| 5 | **Multi-Query** | Search with N paraphrases, keep the unique union |
| 6 | **RAG-Fusion** | Search with N paraphrases, merge the lists by rank (RRF) |
| 7 | **Decomposition** | Split into sub-questions, answer each, synthesize |
| 8 | **Step-Back** | Search with the original **and** a more general question |
| 9 | **HyDE** | Search with a hypothetical answer instead of the question |

Everything still runs locally on the same open stack as Phase 1 — no API keys, no LangChain, no hosted model.

**How to read this notebook.** Each markdown cell explains one idea, and the code cell right below it does
exactly that and prints what happened. Run a cell, look at its output, then read on. Cells build on each other in
order, top to bottom.

## Setup

The Python packages are already installed in this notebook's environment (see `requirements.txt`). The one piece
that is not pip-installed is **Ollama**, the local model runtime, which is installed separately.

Parts 5-9 use that language model **twice per question** — once to write the search queries, once to write the
answer — which is the first time the model sits inside the retrieval path.

In [36]:
# From a terminal, if you ever need to reinstall:
#     .venv\Scripts\python.exe -m pip install -r requirements.txt
import json
import re

import requests
import trafilatura
import chromadb

import ollama
from chonkie import RecursiveChunker
from chonkie.tokenizer import TransformersAutoTokenizer
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
from tokenizers import Tokenizer

print('imports ok')

imports ok


### Configuration

Every tunable lives in one `CONFIG` dict, so experimenting never means editing the pipeline code. Most keys carry
over from Phase 1; the ones at the bottom are new, because we now generate queries.

In [37]:
CONFIG = {
    # --- the source document ---
    'url': 'https://lilianweng.github.io/posts/2023-06-23-agent/',
    'source_id': 'lilianweng',
    'collection': 'lilianweng',

    # --- indexing (as in Phase 1) ---
    'chunk_size_tokens': 300,
    'embed_model': 'Qwen/Qwen3-Embedding-0.6B',
    'store_path': './index/chroma',

    # --- retrieval and generation (as in Phase 1) ---
    'k': 5,
    'llm_model': 'gpt-oss:20b',
    'temperature': 0.0,
    'context_window': 32768,
    'max_answer_tokens': 1024,

    # --- new in Phase 2 ---
    'num_query_variants': 5,    # Part 5, Multi-Query
    'num_fusion_queries': 4,     # Part 6, RAG-Fusion
    'num_sub_questions': 3,      # Part 7, Decomposition
    'rrf_k': 60,                 # Part 6, the RRF constant
    'max_context_chunks': 10,    # cap on chunks handed to the model
    'max_query_tokens': 512,     # cap on a generated search query
    'think': False,              # gpt-oss:20b: no reasoning transcript
}
CONFIG

{'url': 'https://lilianweng.github.io/posts/2023-06-23-agent/',
 'source_id': 'lilianweng',
 'collection': 'lilianweng',
 'chunk_size_tokens': 300,
 'embed_model': 'Qwen/Qwen3-Embedding-0.6B',
 'store_path': './index/chroma',
 'k': 5,
 'llm_model': 'gpt-oss:20b',
 'temperature': 0.0,
 'context_window': 32768,
 'max_answer_tokens': 1024,
 'num_query_variants': 5,
 'num_fusion_queries': 4,
 'num_sub_questions': 3,
 'rrf_k': 60,
 'max_context_chunks': 10,
 'max_query_tokens': 512,
 'think': False}

### Check the local model runtime

`gpt-oss:20b` is a **reasoning** model: left alone it may answer with a transcript of its own thinking, which
would be useless as a search query. Parts 5–9 therefore call it with thinking switched off, and this cell
confirms the runtime is up and the model is present.

One caveat worth knowing before you trust a local reasoning model: on this ollama build the `think=False`
flag does **not** stop the reasoning channel — it still runs, and still comes back as `message.thinking`. The
visible consequence is that a short job like "rewrite this question" can spend its whole token budget on
thinking and return an **empty** answer. `llm()` below detects that and retries with a bigger budget, because an
empty reply is the kind of failure that otherwise shows up later as a mysteriously unhelpful search query.

Even with thinking off, the model wraps a lot of replies in a small JSON envelope such as
`{"thoughts": {"text": ...}}`. `clean_answer()` unwraps that so the answers below read as plain prose.

In [38]:
try:
    listing = ollama.list()
    names = [m.get('name', m.get('model', '')) for m in listing.get('models', [])]
    print('Ollama is running. Available models:')
    for n in names:
        print(' -', n)
    present = any(CONFIG['llm_model'].split(':')[0] in n for n in names)
    print('Configured model present:', present)
    if not present:
        print('Pull it with:  ollama pull', CONFIG['llm_model'])
except Exception as e:
    print('Could not reach Ollama:', e)
    print('Start it with:  ollama serve   then   ollama pull', CONFIG['llm_model'])

Ollama is running. Available models:
 - gpt-oss:20b
Configured model present: True


In [39]:
# One embedding model for everything: indexing, the question, and every query we generate.
model = SentenceTransformer(CONFIG['embed_model'])
tokenizer = Tokenizer.from_pretrained(CONFIG['embed_model'])

print('embedding model:', CONFIG['embed_model'])
print('vector length:  ', model.get_sentence_embedding_dimension())

## Build the index

Query transformations are only ever as good as what they search, so the store itself comes first.

Phase 1 kept its chunks in a collection called `docs`, and — because its overview cell indexed the same article
a second time — that collection ended up holding near-duplicate chunks. You can see them arriving together in
its top-k output. So this notebook builds a **clean, source-scoped** collection instead: the article is indexed
once, into `lilianweng`, and the collection is deleted before it is rebuilt so that repeated runs give exactly
the same store.

Two honest notes on chunking, so this store is genuinely the same one Phase 1 built:

- The chunks are **token-sized**, using the embedding model's own tokenizer — exactly as in Phase 1. That
  matters, because `chonkie`'s `RecursiveChunker` defaults to a *character* tokenizer when you do not pass one,
  and a character budget of 300 would cut this article into 185 pieces instead of 34.
- In 1.7.0 `RecursiveChunker` exposes no overlap parameter, so these chunks are packed back to back rather
  than overlapping. Phase 1 used `TokenChunker` for its overlap demo; the retrieval pipeline itself never
  overlapped.

In [40]:
def build_index():
    """Load the article, split it, embed every chunk, store them in a clean collection."""
    raw_html = requests.get(CONFIG['url'], timeout=30).text
    text = trafilatura.extract(raw_html) or ''
    if not text.strip():
        raise RuntimeError(
            f"trafilatura extracted no text from {CONFIG['url']}. The page may have moved, "
            f"or the markup changed. Fix the URL or the extraction before continuing."
        )

    # Token-sized chunks, using the embedding model's own tokenizer — the same chunker Phase 1
    # used. Without an explicit tokenizer RecursiveChunker falls back to a *character* tokenizer,
    # which would silently make chunk_size_tokens a character budget. There is no overlap knob
    # in 1.7.0, so these chunks are packed back to back.
    auto_tok = AutoTokenizer.from_pretrained(CONFIG['embed_model'])
    chonkie_tok = TransformersAutoTokenizer(auto_tok)
    chunker = RecursiveChunker(
        tokenizer=chonkie_tok, chunk_size=CONFIG['chunk_size_tokens']
    )
    passages = [c.text for c in chunker.chunk(text)]
    print(f'loaded {len(text)} characters -> {len(passages)} chunks')

    # `count_tokens` is defined in the helpers cell further down, so count with a local
    # tokenizer here to keep this cell self-contained.
    raw_tok = Tokenizer.from_pretrained(CONFIG['embed_model'])
    sizes = [len(raw_tok.encode(p).ids) for p in passages]
    print(f'tokens per chunk: min {min(sizes)} / mean {sum(sizes) / len(sizes):.0f} / max {max(sizes)}')
    if max(sizes) > CONFIG['chunk_size_tokens'] * 1.5:
        print('WARNING: a chunk is far larger than chunk_size_tokens — check the tokenizer.')

    vectors = model.encode(passages, normalize_embeddings=True)

    client = chromadb.PersistentClient(path=CONFIG['store_path'])
    try:
        client.delete_collection(name=CONFIG['collection'])
    except Exception:
        pass  # nothing there to delete on the first run

    coll = client.create_collection(
        name=CONFIG['collection'], metadata={'hnsw:space': 'cosine'}
    )
    coll.upsert(
        ids=[f"{CONFIG['source_id']}:{i}" for i in range(len(passages))],
        documents=passages,
        embeddings=vectors.tolist(),
        metadatas=[
            {'source_id': CONFIG['source_id'], 'chunk_index': i}
            for i in range(len(passages))
        ],
    )
    return coll


collection = build_index()
print('collection:', collection.name, '-> chunks:', collection.count())

loaded 39316 characters -> 34 chunks
tokens per chunk: min 23 / mean 260 / max 301
collection: lilianweng -> chunks: 34


## The helpers every part shares

All five techniques have the same shape: **build a better search input, then run the normal chain**. So we build
the pieces once and reuse them in every part.

- **`llm(...)`** — one call to the local model, with thinking requested off and a retry if the
  reasoning channel swallows the whole token budget.
- **`clean_answer(...)`** — unwrap the JSON envelope the model wraps its replies in.
- **`retrieve(...)`** — embed a query, return its top-k chunks as plain dicts. Guarded so it can never ask the
  store for more chunks than the store holds.
- **`context_from(...)`** — join chunks into a numbered, cited context block.
- **`answer_question(...)`** — generate a grounded answer, citing `[n]`, declining when unsupported.
- **`parse_queries(...)`** — turn messy model output into a clean list of queries.

Two more decide *how* several hit lists become one: **`unique_union`** keeps each chunk once, and
**`reciprocal_rank_fusion`** merges them *by rank*.

In [41]:
def llm(prompt, system=None, max_tokens=None, _retried=False):
    """One call to the local model.

    Robustness note, learned the hard way: although we pass `think=False`, this ollama build
    still runs gpt-oss's reasoning channel and returns it as `message.thinking`. On a short
    task that reasoning can consume the entire `num_predict` budget, leaving
    `message.content` **empty**. An empty reply is not an error — it silently becomes a
    fallback question or a blank answer — so we detect it and retry once with room to spare.
    """
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})

    budget = max_tokens or CONFIG['max_answer_tokens']
    options = {
        'temperature': CONFIG['temperature'],
        'num_predict': budget,
    }

    try:
        reply = ollama.chat(
            model=CONFIG['llm_model'],
            messages=messages,
            options=options,
            think=CONFIG['think'],
        )
    except TypeError:
        # Older ollama clients do not accept a `think` argument.
        reply = ollama.chat(
            model=CONFIG['llm_model'], messages=messages, options=options
        )

    content = (reply.get('message', {}).get('content') or '').strip()

    # The reasoning channel ate the whole budget. Give the answer room and try once more.
    if not content and not _retried:
        return llm(prompt, system=system, max_tokens=budget * 4, _retried=True)

    return content


def clean_answer(text):
    """Unwrap the JSON envelope gpt-oss puts around most replies.

    With `think=False` the model usually still replies as JSON, e.g.
    {"thoughts": {"text": "..."}}. For a teaching notebook the useful part is the prose, so
    lift `text` out when we can and otherwise return the reply untouched. Query-generation
    calls do NOT go through here: `parse_queries` needs the raw lines.
    """
    stripped = (text or '').strip()
    if not stripped.startswith('{'):
        return stripped
    try:
        payload = json.loads(stripped)
    except (ValueError, TypeError):
        return stripped
    if not isinstance(payload, dict):
        return stripped
    thoughts = payload.get('thoughts')
    for candidate in (thoughts, payload):
        if isinstance(candidate, dict):
            value = candidate.get('text') or candidate.get('speak') or candidate.get('summary')
            if isinstance(value, str) and value.strip():
                return value.strip()
    return stripped

Every answer below goes through **`clean_answer()`** so the model's JSON envelope does not drown the prose.
**`count_tokens(...)`** reuses the same tokenizer, and **`context_from(...)`** numbers the chunks so the model
can cite them the way Phase 1 did.

In [42]:
def retrieve(query, k=None):
    """Embed a query and return its nearest chunks as plain dicts."""
    k = k or CONFIG['k']
    total = collection.count()
    if total == 0:
        return []

    vector = model.encode([query], normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=vector, n_results=min(k, total))

    hits = []
    for cid, doc, meta, dist in zip(
        result['ids'][0], result['documents'][0],
        result['metadatas'][0], result['distances'][0],
    ):
        hits.append({
            'id': cid,
            'text': doc,
            'source_id': (meta or {}).get('source_id'),
            'score': 1 - dist,  # the store returns cosine distance; similarity is its inverse
        })
    return hits

def context_from(hits, limit=None):
    """Join chunk texts as a numbered, cited context block, best first.

    The `[n]` markers match Phase 1's citation scheme, so an answer can point at the exact
    chunk it used and `print_sources` can map that number back to a chunk id.
    """
    if not hits:
        return ''
    limit = limit or CONFIG['max_context_chunks']
    return '\n\n'.join(
        f'[{i}] {hit["text"]}' for i, hit in enumerate(hits[:limit], start=1)
    )


def print_sources(hits, limit=None):
    """Show the numbered context blocks the model was given, with their scores and ids."""
    if not hits:
        print('(no chunks retrieved)')
        return
    limit = limit or CONFIG['max_context_chunks']
    print('Sources available to the model:')
    for i, hit in enumerate(hits[:limit], start=1):
        print(f'  [{i}] score={hit["score"]:.3f}  id={hit["id"]}')


def count_tokens(text):
    """Token count, using the embedding model's own tokenizer."""
    return len(tokenizer.encode(text).ids)


def answer_question(question, context, max_tokens=None):
    """Generate an answer grounded only in the given context.

    Same contract as Phase 1's generation step: answer only from the context, cite the `[n]`
    blocks you used, and say plainly when the context does not contain the answer.
    """
    prompt = f"""Answer the question using only the context below.
Cite the context blocks you use as [n]. If the context does not contain the answer, say so plainly.

Context:
{context}

Question: {question}
"""
    return clean_answer(llm(prompt, max_tokens=max_tokens))

In [43]:
def parse_queries(raw, n, fallback=None):
    """Turn model output into a clean list of at most n queries.

    Handles numbered and bulleted lists, stray headings, duplicates and blank lines, so the
    techniques never depend on the model formatting its reply exactly as asked.
    """
    cleaned, seen = [], set()

    for line in raw.splitlines():
        line = re.sub(r'^\(?\d+\)?[.)]\s*', '', line.strip())   # '1. '  '2) '  '(3) '
        line = re.sub(r'^[-*\u2022]\s*', '', line)              # '- '  '* '  bullet
        line = line.strip().strip('"').strip()

        if len(line) < 3:            # blank, or too short to be a real query
            continue
        if line.endswith(':'):       # a stray heading, e.g. 'Here are the queries:'
            continue
        if line.lower() in seen:     # the same query twice
            continue

        seen.add(line.lower())
        cleaned.append(line)
        if len(cleaned) == n:
            break

    return cleaned or ([fallback] if fallback else [])


def generate_queries(question, template, n, system=None):
    """Ask the model for n alternative search queries."""
    raw = llm(
        template.format(question=question),
        system=system,
        max_tokens=CONFIG['max_query_tokens'],
    )
    return parse_queries(raw, n, fallback=question)


def unique_union(result_lists):
    """Merge several hit lists, keeping each chunk only once."""
    seen, union = set(), []
    for hits in result_lists:
        for hit in hits:
            if hit['id'] not in seen:
                seen.add(hit['id'])
                union.append(hit)
    return union

In [44]:
def reciprocal_rank_fusion(result_lists, k=None):
    """Merge several *ranked* lists by Reciprocal Rank Fusion.

    A chunk found at rank r in one list earns 1 / (rank + k) points for that list, so a chunk
    that several queries agree on accumulates points and floats to the top. Ranks are 0-based and
    k defaults to 60; the scores are only ever compared with each other.
    """
    k = k or CONFIG['rrf_k']
    scores, best = {}, {}

    for hits in result_lists:
        for rank, hit in enumerate(hits):
            cid = hit['id']
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (rank + k)
            best.setdefault(cid, hit)  # keep one copy of the chunk text

    fused = sorted(scores.items(), key=lambda item: (-item[1], item[0]))
    return [(best[cid], score) for cid, score in fused]

## The baseline we are improving on

First, Phase 1's chain, unchanged: embed the question exactly as written, take the top `k` chunks, answer from
them. `retrieve`, `context_from` and `answer_question` are the same three steps Phase 1 took, and this cell is
that pipeline verbatim — the same `rag(question)` from its "Putting it together" section.

Everything below alters *only* the search input. The store is the same one, the numbered context and the
generation prompt are the same, and the model never changes, which is what makes the differences comparable.
The one thing to hold onto: any difference you see from here on came from changing **the string that got
embedded**.

In [45]:
QUESTION = 'What is Task Decomposition?'

baseline_hits = retrieve(QUESTION)

print(f'top-{len(baseline_hits)} for the raw question:')
for hit in baseline_hits:
    print(f"  {hit['id']}  score={hit['score']:.3f}  {hit['text'][:70]}...")

print()
print('context tokens:', count_tokens(context_from(baseline_hits)))
print()
print_sources(baseline_hits)

top-5 for the raw question:
  lilianweng:1  score=0.641  A complicated task usually involves many steps. An agent needs to know...
  lilianweng:21  score=0.602          "reasoning": "reasoning",
        "plan": "- short bulleted\n-...
  lilianweng:22  score=0.569      "content": "We are writing {{a Super Mario game in python. MVC com...
  lilianweng:20  score=0.569  14. Get Improved Code: "improve_code", args: "suggestions": "<list_of_...
  lilianweng:19  score=0.545  2. Browse Website: "browse_website", args: "url": "<url>", "question":...

context tokens: 1390

Sources available to the model:
  [1] score=0.641  id=lilianweng:1
  [2] score=0.602  id=lilianweng:21
  [3] score=0.569  id=lilianweng:22
  [4] score=0.569  id=lilianweng:20
  [5] score=0.545  id=lilianweng:19


In [46]:
def baseline_rag(question, k=None):
    """The Phase 1 pipeline: one query, its top-k chunks, one answer."""
    return answer_question(question, context_from(retrieve(question, k)))


print(baseline_rag(QUESTION))

Task Decomposition is the process of breaking a complex task into smaller, manageable steps or sub‑goals so that an agent can plan and execute them more easily. It is often achieved through prompting techniques such as Chain of Thought (CoT) or Tree of Thoughts, and can be performed by an LLM with simple prompts, by using task‑specific instructions, or with human input. [1]


# Part 5 — Multi-Query

One question becomes one point in a very large vector space, and a question is rarely phrased the way its best
passage is. **Multi-Query** asks the model for several perspectives on the same question, retrieves once per
perspective, and keeps the **unique union** of the results.

It is a fan-out: one question in, several retrievals, one merged list out.

```
question
   |
   +--> paraphrase 1 --> top-k ----+
   +--> paraphrase 2 --> top-k ----+
   +--> paraphrase 3 --> top-k ----+--> unique union --> answer
   +--> ...            --> top-k ---+
```

In [47]:
MULTI_QUERY_PROMPT = (
    'You are an AI language model assistant. Your task is to generate five different versions of the '
    'given user question to retrieve relevant documents from a vector database. By generating multiple '
    'perspectives on the user question, your goal is to help the user overcome some of the limitations '
    'of the distance-based similarity search. Provide these alternative questions separated by newlines.\n'
    'Original question: {question}'
)

raw = llm(MULTI_QUERY_PROMPT.format(question=QUESTION), max_tokens=CONFIG['max_query_tokens'])

print('--- raw model output ---')
print(raw)
print()
print('--- after parse_queries ---')
for i, q in enumerate(parse_queries(raw, CONFIG['num_query_variants'], fallback=QUESTION), start=1):
    print(f'[{i}] {q}')

--- raw model output ---
What does the term “Task Decomposition” mean in project management?  
Can you explain the concept of task decomposition and its key components?  
How is task decomposition used to break down complex projects into manageable parts?  
What are the benefits and typical steps involved in task decomposition?  
In what contexts is task decomposition applied, and can you give an example?

--- after parse_queries ---
[1] What does the term “Task Decomposition” mean in project management?
[2] Can you explain the concept of task decomposition and its key components?
[3] How is task decomposition used to break down complex projects into manageable parts?
[4] What are the benefits and typical steps involved in task decomposition?
[5] In what contexts is task decomposition applied, and can you give an example?


In [48]:
queries = parse_queries(raw, CONFIG['num_query_variants'], fallback=QUESTION)

result_lists = []
for q in queries:
    hits = retrieve(q)
    result_lists.append(hits)
    print(f'--- retrieved for: {q}')
    for hit in hits:
        print(f"    {hit['id']}  score={hit['score']:.3f}")

--- retrieved for: What does the term “Task Decomposition” mean in project management?
    lilianweng:1  score=0.551
    lilianweng:21  score=0.478
    lilianweng:22  score=0.456
    lilianweng:20  score=0.429
    lilianweng:19  score=0.417
--- retrieved for: Can you explain the concept of task decomposition and its key components?
    lilianweng:1  score=0.633
    lilianweng:21  score=0.520
    lilianweng:22  score=0.504
    lilianweng:20  score=0.478
    lilianweng:27  score=0.459
--- retrieved for: How is task decomposition used to break down complex projects into manageable parts?
    lilianweng:1  score=0.612
    lilianweng:21  score=0.543
    lilianweng:22  score=0.509
    lilianweng:20  score=0.487
    lilianweng:19  score=0.471
--- retrieved for: What are the benefits and typical steps involved in task decomposition?
    lilianweng:1  score=0.653
    lilianweng:21  score=0.583
    lilianweng:20  score=0.553
    lilianweng:22  score=0.545
    lilianweng:27  score=0.502
--- retri

In [49]:
union = unique_union(result_lists)
union.sort(key=lambda hit: hit['score'], reverse=True)

print(f"{len(queries)} queries x k={CONFIG['k']} = {len(queries) * CONFIG['k']} chunks retrieved")
print(f'unique chunks in the union: {len(union)}')
print()
for hit in union:
    print(f"  {hit['id']}  score={hit['score']:.3f}  {hit['text'][:70]}...")

5 queries x k=5 = 25 chunks retrieved
unique chunks in the union: 7

  lilianweng:1  score=0.551  A complicated task usually involves many steps. An agent needs to know...
  lilianweng:21  score=0.478          "reasoning": "reasoning",
        "plan": "- short bulleted\n-...
  lilianweng:27  score=0.459   #  … same conversation as earlier, ended with "Make your own assumpti...
  lilianweng:22  score=0.456      "content": "We are writing {{a Super Mario game in python. MVC com...
  lilianweng:20  score=0.429  14. Get Improved Code: "improve_code", args: "suggestions": "<list_of_...
  lilianweng:19  score=0.417  2. Browse Website: "browse_website", args: "url": "<url>", "question":...
  lilianweng:12  score=0.395  ChatGPT Plugins and OpenAI API function calling are good examples of L...


The union is already larger than the baseline context, and it grows with both the number of queries and `k`. That
is the first cost of query transformation: more retrieved text, and a bigger bill to fit into the context
window. `max_context_chunks` is the guard; note the token counts either side of it, the share of the window the
capped context uses, and the worst case before deduplication. That last number is the one to remember: five
queries at `k=5` is 25 chunks requested, and deduplication only narrows the gap by however much the queries
happened to agree. Raise the query count and you raise that ceiling, not the average.

In [50]:
union_context = context_from(union)                       # already capped at max_context_chunks
uncapped_context = '\n\n'.join(hit['text'] for hit in union)

union_tokens = count_tokens(uncapped_context)
capped_tokens = count_tokens(union_context)
window = CONFIG['context_window']

print(f"baseline context: {len(baseline_hits)} chunks, {count_tokens(context_from(baseline_hits))} tokens")
print(f"union context:    {len(union)} chunks, {union_tokens} tokens  (uncapped)")
print(f"capped context:   {len(union[:CONFIG['max_context_chunks']])} chunks, "
      f"{capped_tokens} tokens  (max_context_chunks={CONFIG['max_context_chunks']})")
print(f"context window:   {window} tokens -> the capped context uses "
      f"{100 * capped_tokens / window:.1f}% of it")

# The arithmetic that makes the ceiling necessary: N queries x k chunks is the worst case,
# deduplication only brings it back down to what the queries happened to agree on.
print()
print(f"worst case {len(result_lists)} queries x k={CONFIG['k']} = "
      f"{len(result_lists) * CONFIG['k']} chunks, before deduplication")
if union_tokens > window:
    print(f"WARNING: the merged context is {union_tokens - window} tokens OVER the window. "
          f"Lower max_context_chunks, or the number of queries.")

baseline context: 5 chunks, 1390 tokens
union context:    7 chunks, 1789 tokens  (uncapped)
capped context:   7 chunks, 1813 tokens  (max_context_chunks=10)
context window:   32768 tokens -> the capped context uses 5.5% of it

worst case 5 queries x k=5 = 25 chunks, before deduplication


In [51]:
print(answer_question(QUESTION, union_context))

Task Decomposition is the process of breaking a complex task into smaller, manageable steps or sub‑goals so that an agent can plan and execute them more easily. It is often achieved by prompting an LLM to list steps (e.g., “Steps for XYZ”), using task‑specific instructions (e.g., “Write a story outline”), or by human input. Techniques such as Chain of Thought (CoT) and Tree of Thoughts extend this idea by encouraging step‑by‑step reasoning and exploring multiple reasoning paths at each step. [1]


# Part 6 — RAG-Fusion

Multi-Query throws away the one piece of information the store gave us for free: **the order it ranked things
in**. If a chunk was the very best hit for two of the four queries, the plain union has no way to know that.

**RAG-Fusion** keeps the rank. Each query still retrieves its own top-k, and every chunk found at rank `r` earns

```
score(chunk) += 1 / (rank + k)
```

for each list it appears in. Chunks that several queries agree on accumulate points and rise to the top. The
same four queries in, a *better ordered* list out.

In [52]:
FUSION_PROMPT = (
    'You are a helpful assistant that generates multiple search queries based on a single input query.\n'
    'Generate multiple search queries related to: {question}\n'
    'Output (4 queries):'
)

fusion_queries = generate_queries(QUESTION, FUSION_PROMPT, CONFIG['num_fusion_queries'])

for i, q in enumerate(fusion_queries, start=1):
    print(f'[{i}] {q}')

[1] “Task decomposition definition and practical examples”
[2] “Step‑by‑step guide to task decomposition in project management”
[3] “Task decomposition techniques used in AI and machine learning”
[4] “Benefits of task decomposition for team productivity and workflow”


In [53]:
fusion_lists = [retrieve(q) for q in fusion_queries]

for q, hits in zip(fusion_queries, fusion_lists):
    print(f'--- retrieved for: {q}')
    for hit in hits:
        print(f"    {hit['id']}  score={hit['score']:.3f}")

--- retrieved for: “Task decomposition definition and practical examples”
    lilianweng:21  score=0.651
    lilianweng:1  score=0.634
    lilianweng:20  score=0.624
    lilianweng:22  score=0.597
    lilianweng:19  score=0.594
--- retrieved for: “Step‑by‑step guide to task decomposition in project management”
    lilianweng:21  score=0.631
    lilianweng:22  score=0.593
    lilianweng:19  score=0.592
    lilianweng:20  score=0.577
    lilianweng:1  score=0.564
--- retrieved for: “Task decomposition techniques used in AI and machine learning”
    lilianweng:21  score=0.624
    lilianweng:1  score=0.618
    lilianweng:20  score=0.609
    lilianweng:33  score=0.606
    lilianweng:19  score=0.595
--- retrieved for: “Benefits of task decomposition for team productivity and workflow”
    lilianweng:21  score=0.605
    lilianweng:20  score=0.601
    lilianweng:19  score=0.570
    lilianweng:1  score=0.554
    lilianweng:22  score=0.532

    lilianweng:21  score=0.651
    lilianweng:1  score=

In [54]:
fused = reciprocal_rank_fusion(fusion_lists, k=CONFIG['rrf_k'])

print(f'{len(fusion_lists)} ranked lists, {sum(len(h) for h in fusion_lists)} hits, '
      f'{len(fused)} unique chunks\n')
for rank, (hit, score) in enumerate(fused, start=1):
    print(f'{rank:>2}. {hit["id"]}  rrf={score:.5f}  best_rank_score={hit["score"]:.3f}  {hit["text"][:60]}...')

4 ranked lists, 20 hits, 6 unique chunks

 1. lilianweng:21  rrf=0.06667  best_rank_score=0.651          "reasoning": "reasoning",
        "plan": "- short b...
 2. lilianweng:20  rrf=0.06452  best_rank_score=0.624  14. Get Improved Code: "improve_code", args: "suggestions": ...
 3. lilianweng:1  rrf=0.06428  best_rank_score=0.634  A complicated task usually involves many steps. An agent nee...
 4. lilianweng:19  rrf=0.06351  best_rank_score=0.594  2. Browse Website: "browse_website", args: "url": "<url>", "...
 5. lilianweng:22  rrf=0.04789  best_rank_score=0.597      "content": "We are writing {{a Super Mario game in pytho...
 6. lilianweng:33  rrf=0.01587  best_rank_score=0.606  [21] GPT-Engineer. https://github.com/AntonOsika/gpt-enginee...


Two things are worth reading off that ranking. A chunk appearing in several lists sums several small scores and
jumps; a chunk that only ever ranked low stays low even though a plain union would have kept it. RRF is also the
reason this is not simply "take the best of the best": no single query's ordering is trusted on its own.

Note the fused list is already ordered, and `context_from` truncates while preserving that order, so the
strongest chunks survive the cap.

In [55]:
fused_context = context_from([hit for hit, _ in fused])

print('context tokens:', count_tokens(fused_context))
print()
print(answer_question(QUESTION, fused_context))

context tokens: 1416

Task decomposition is the process of breaking a complex task into smaller, manageable steps or sub‑goals, enabling an agent to plan and execute each part separately. [3]


# Part 7 — Decomposition

Every technique so far still asks one question of the store. **Decomposition** changes the shape of the question:
it breaks it into **sub-questions that can each be answered on their own**, retrieves for each, and combines the
results. It is the right tool when one question is really several questions wearing a trench coat — *what are
the main components of an LLM-powered agent system?* is really a request for a list of parts.

There are two honest ways to combine the sub-answers, and the notebook builds both.

In [56]:
DECOMPOSITION_PROMPT = (
    'You are a helpful assistant that generates multiple sub-questions related to an input question.\n'
    'The goal is to break down the input into a set of sub-problems / sub-questions that can be answered '
    'in isolation.\n'
    'Generate multiple search queries related to: {question}\n'
    'Output (3 queries):'
)

sub_questions = generate_queries(QUESTION, DECOMPOSITION_PROMPT, CONFIG['num_sub_questions'])

for i, q in enumerate(sub_questions, start=1):
    print(f'[{i}] {q}')

[1] “Task decomposition definition and key concepts”
[2] “How to perform task decomposition in project management”
[3] “Examples of task decomposition in software development”


## Variant A — answer recursively

Answer the sub-questions in order, and carry every answer so far as background for the next one. Early answers
become evidence for later ones, which is what makes this *recursive* rather than merely parallel.

The cost is order dependence: a poor first answer can steer the rest, and the accumulated background grows the
prompt on every step.

In [57]:
def recursive_answers(question, sub_questions, max_tokens=512):
    """Answer each sub-question in turn, carrying earlier answers as background."""
    q_a_pairs = ''
    steps = []

    for sub_q in sub_questions:
        hits = retrieve(sub_q)
        prompt = f"""Here is the question you need to answer:

--- {sub_q} ---

Here is any available background question + answer pairs:

--- {q_a_pairs} ---

Here is additional context relevant to the question:

--- {context_from(hits)} ---

Use the above context and any background question + answer pairs to answer the question: {sub_q}
"""
        answer = clean_answer(llm(prompt, max_tokens=max_tokens))
        steps.append((sub_q, answer))
        q_a_pairs += f'Question: {sub_q}\nAnswer: {answer}\n\n'

    return steps


recursive_steps = recursive_answers(QUESTION, sub_questions)

for i, (q, a) in enumerate(recursive_steps, start=1):
    print(f'--- step {i}: {q}')
    print(a)
    print()

--- step 1: “Task decomposition definition and key concepts”
**Task Decomposition – Definition & Key Concepts**

**Definition**  
Task decomposition is the systematic process of breaking a complex,

--- step 2: “How to perform task decomposition in project management”
{"answer":"Task decomposition in project management is the systematic process of breaking a project’s scope into smaller, manageable components that can be planned, executed, and monitored more effectively. Below is a practical guide on how to perform task decomposition:\n\n1. **Define the Project Scope**\n   - Clarify objectives, deliverables, and success criteria.\n   - Document the high‑level requirements and constraints.\n\n2. **Create a Work Breakdown Structure (WBS)**\n   - **Top‑down approach**: Start with the final deliverable and decompose it into major

--- step 3: “Examples of task decomposition in software development”




## Variant B — answer individually

Now the same sub-questions, but each one is answered in isolation: its own retrieval, its own context, no memory
of the siblings. No answer can contaminate another, and nothing grows — at the cost of the later steps being
unable to use what the earlier ones found.

Whichever variant you pick, the final step is the same: hand the question-and-answer pairs to the model and ask
for one synthesized answer to the original question.

One caution worth stating now: that last synthesis sees **model-written text**, not raw documents. It can
comfortably fill gaps from the model's own memory — inventing a work breakdown structure, a tool name, an API —
that is in no chunk. That is why the synthesis cell also passes the retrieved context the sub-answers came from,
and why the control question later matters so much.

In [58]:
def individual_answers(question, sub_questions, max_tokens=512):
    """Answer each sub-question on its own, with no knowledge of the others."""
    steps = []
    for sub_q in sub_questions:
        hits = retrieve(sub_q)
        steps.append((sub_q, answer_question(sub_q, context_from(hits), max_tokens=max_tokens)))
    return steps


individual_steps = individual_answers(QUESTION, sub_questions)

for i, (q, a) in enumerate(individual_steps, start=1):
    print(f'--- {q}')
    print(a)
    print()

--- “Task decomposition definition and key concepts”
Task decomposition is the process of breaking a complex task into a series of smaller, more manageable steps or sub‑goals so that each can be tackled individually. It allows an agent (or a human) to plan ahead, understand the sequence of actions required, and reason about each part before moving on to the next.

Key concepts in task decomposition (as described in the context) include:

1. **Chain of Thought (CoT)** – a prompting technique where the model is instructed to “think step by step,” using additional test‑time computation to split a hard problem into simpler sub‑tasks.  
2. **Tree of Thoughts** – an extension of CoT that explores multiple reasoning paths at each step, creating a tree structure. The search can be performed with breadth‑first search (BFS) or depth‑first search (DFS), and each state is evaluated by a classifier or majority vote.  
3. **Subgoal Identification** – determining the intermediate objectives that lead

In [59]:
def synthesize(question, steps, evidence=None, max_tokens=None):
    """Turn a set of question/answer pairs into one answer to the original question.

    `evidence` is the retrieved context the sub-answers came from. Passing it matters: without
    it the final synthesis sees only model-written text and can drift into content the corpus
    never contained, which is exactly the grounding failure RAG exists to prevent.
    """
    pairs = '\n\n'.join(
        f'Question {i}: {q}\nAnswer {i}: {a}'
        for i, (q, a) in enumerate(steps, start=1)
    )
    evidence_block = ''
    if evidence:
        evidence_block = f"""
Here is the retrieved context the answers above were drawn from:

{evidence}
"""
    prompt = f"""Here is a set of Q+A pairs:

{pairs}
{evidence_block}
Use these to synthesize an answer to the question: {question}
Answer using only the context and Q+A pairs above. Cite the context blocks you use as [n].
"""
    return clean_answer(llm(prompt, max_tokens=max_tokens))


# The context behind the sub-answers, so the synthesis above stays grounded in real chunks.
sub_evidence = unique_union([retrieve(q) for q in sub_questions])
print('--- synthesized from the individually answered sub-questions ---')
print(synthesize(QUESTION, individual_steps, evidence=context_from(sub_evidence)))

--- synthesized from the individually answered sub-questions ---



# Part 8 — Step-Back

A question can be too specific to retrieve with. Ask *when was Jan Sindel born* and you search for a date;
ask *what is Jan Sindel's personal history* and you search for the passage that actually contains it.

**Step-Back** paraphrases the question into a broader one, then retrieves with **both** — the original for
precision, the step-back question for coverage. The two contexts are then labelled and handed over together, so
the model can tell which came from where.

The paraphrase is done with few-shot examples rather than instructions alone: showing two hand-written
input/output pairs is far more reliable than describing the transformation you want.

One practical warning before the code: this technique needs a question that is **specific enough to step back
from**. Hand it the running `What is Task Decomposition?` and the model has nothing to generalise — it will
return the same question, both retrievals will be identical, and the technique will appear to do nothing. The
cell below therefore demonstrates the rewrite on a narrower question from the same article.

In [60]:
STEP_BACK_PROMPT = """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:

Question: Could the members of The Police perform lawful arrests?
Step-back question: what can the members of The Police do?

Question: Jan Sindel's was born in what country?
Step-back question: what is Jan Sindel's personal history?

Question: {question}
Step-back question:"""


def step_back_question(question):
    """Paraphrase a question into a broader one that is easier to retrieve with."""
    raw = llm(STEP_BACK_PROMPT.format(question=question), max_tokens=CONFIG['max_query_tokens'])

    # The model sometimes echoes the whole few-shot block; keep only what follows the last marker.
    if 'Step-back question:' in raw:
        raw = raw.rsplit('Step-back question:', 1)[1]

    parsed = parse_queries(raw, 1, fallback=question)
    return parsed[0].strip().strip('"') if parsed else question


# A deliberately narrow question: step-back only does something when there is something to
# step back from. Watch `changed` — if it reads False, the rewrite was a no-op and the two
# retrievals below will be identical.
STEP_BACK_DEMO = 'How does Chain of Thought help an agent break a task into smaller steps?'
step_q = step_back_question(STEP_BACK_DEMO)
changed = step_q.strip().lower() != STEP_BACK_DEMO.strip().lower()

print('original:  ', STEP_BACK_DEMO)
print('step-back: ', step_q)
print('changed:   ', changed)
if not changed:
    print('-> the model returned the question unchanged. Step-back needs a specific input;')
    print('   re-run this cell with a narrower question to see the technique do its work.')

original:   How does Chain of Thought help an agent break a task into smaller steps?
step-back:  How do agents break tasks into smaller steps?
changed:    True


In [61]:
normal_hits = retrieve(STEP_BACK_DEMO)
step_hits = retrieve(step_q)

print(f'--- original question: {STEP_BACK_DEMO}')
for hit in normal_hits:
    print(f"    {hit['id']}  score={hit['score']:.3f}  {hit['text'][:60]}...")

print()
print(f'--- step-back question: {step_q}')
for hit in step_hits:
    print(f"    {hit['id']}  score={hit['score']:.3f}  {hit['text'][:60]}...")

print()
print('chunks only the step-back question found:')
extra = [hit['id'] for hit in step_hits if hit['id'] not in {h['id'] for h in normal_hits}]
print('  ', extra or '(none — both questions found the same chunks)')

--- original question: How does Chain of Thought help an agent break a task into smaller steps?
    lilianweng:1  score=0.727  A complicated task usually involves many steps. An agent nee...
    lilianweng:21  score=0.652          "reasoning": "reasoning",
        "plan": "- short b...
    lilianweng:20  score=0.623  14. Get Improved Code: "improve_code", args: "suggestions": ...
    lilianweng:30  score=0.605    author  = "Weng, Lilian",
  journal = "lilianweng.github.i...
    lilianweng:19  score=0.596  2. Browse Website: "browse_website", args: "url": "<url>", "...

--- step-back question: How do agents break tasks into smaller steps?
    lilianweng:1  score=0.645  A complicated task usually involves many steps. An agent nee...
    lilianweng:21  score=0.577          "reasoning": "reasoning",
        "plan": "- short b...
    lilianweng:20  score=0.559  14. Get Improved Code: "improve_code", args: "suggestions": ...
    lilianweng:22  score=0.551      "content": "We are writing {{a 

In [62]:
# Both sets of context, each labelled so the model knows what it is looking at.
# The answer is still given the running QUESTION — step-back only ever changed the search.
step_context = (
    '# Context retrieved with the original question:\n'
    f'{context_from(normal_hits)}\n\n'
    f'# Context retrieved with the step-back question ({step_q}):\n'
    f'{context_from(step_hits)}'
)

print('context tokens:', count_tokens(step_context))
print()
print_sources(normal_hits + step_hits, limit=CONFIG['k'])
print()
print(answer_question(QUESTION, step_context))

context tokens: 2705

Sources available to the model:
  [1] score=0.727  id=lilianweng:1
  [2] score=0.652  id=lilianweng:21
  [3] score=0.623  id=lilianweng:20
  [4] score=0.605  id=lilianweng:30
  [5] score=0.596  id=lilianweng:19

Task decomposition is the process of breaking a complex task into smaller, manageable steps so that an agent can understand what needs to be done and plan ahead. It often involves techniques such as Chain‑of‑Thought prompting, Tree‑of‑Thoughts, or simple LLM prompts that ask for sub‑goals or steps. [1]


# Part 9 — HyDE

A question and the passage that answers it rarely look alike. *"What is task decomposition?"* shares almost no
words with *"Task decomposition can be done by LLM with simple prompting, or by task-specific instructions"*.

**HyDE** (Hypothetical Document Embeddings) inverts the usual direction: instead of searching with the question,
ask the model to **write a short passage that would answer it**, then search with *that*. The fake passage looks
like the real one, so it lands near the right chunks even though it is entirely invented.

> The hypothetical passage is a **query**, never an answer. It is written from the model's own memory, so it is
> ungrounded by construction — and it is never shown to anyone as a result. Only its embedding is used, to point
> the search at the right place. The answer still comes from retrieved, real chunks.

In [63]:
HYDE_PROMPT = (
    'Please write a scientific paper passage to answer the question.\n'
    'Question: {question}\n'
    'Passage:'
)

hypothetical = llm(HYDE_PROMPT.format(question=QUESTION), max_tokens=400)

print(hypothetical)

In [64]:
hyde_hits = retrieve(hypothetical)

print(f'retrieved with the hypothetical passage ({count_tokens(hypothetical)} tokens) '
      f'instead of the {count_tokens(QUESTION)}-token question:')
for hit in hyde_hits:
    print(f"  {hit['id']}  score={hit['score']:.3f}  {hit['text'][:70]}...")

print()
print('for comparison, the baseline retrieved:')
for hit in baseline_hits:
    print(f"  {hit['id']}  score={hit['score']:.3f}")

retrieved with the hypothetical passage (1 tokens) instead of the 7-token question:
  lilianweng:19  score=0.480  2. Browse Website: "browse_website", args: "url": "<url>", "question":...
  lilianweng:20  score=0.464  14. Get Improved Code: "improve_code", args: "suggestions": "<list_of_...
  lilianweng:22  score=0.463      "content": "We are writing {{a Super Mario game in python. MVC com...
  lilianweng:33  score=0.450  [21] GPT-Engineer. https://github.com/AntonOsika/gpt-engineer...
  lilianweng:21  score=0.438          "reasoning": "reasoning",
        "plan": "- short bulleted\n-...

for comparison, the baseline retrieved:
  lilianweng:1  score=0.641
  lilianweng:21  score=0.602
  lilianweng:22  score=0.569
  lilianweng:20  score=0.569
  lilianweng:19  score=0.545


The question asked of the model at the end is still the **original** question. HyDE only ever influenced the
search.

In [65]:
print(answer_question(QUESTION, context_from(hyde_hits)))

The context does not contain the answer.


### When HyDE makes things worse

Read the two hit lists in the cell above rather than assuming more text is better. HyDE replaces a short,
specific question with a long passage written from the model's memory, and that passage is only useful if it
**looks like the corpus**. It can also drift: asked for a scientific passage about task decomposition, a model
may write about combinatorial planning spaces and modular robotics — fluent, plausible, and pointing at chunks
that do not answer the question.

That is the honest shape of this technique. HyDE helps when the question and the answering passage share
almost no vocabulary, and it *hurts* when the question is already well-aimed, because it trades a precise
needle for a noisy one. The retrieval above is the diagnostic: if the hypothetical passage retrieves chunks
unrelated to the question, discard it. The running comparison at the end of the notebook shows exactly this —
HyDE declining the answer that every other strategy got.

## Putting it together

Strip away the prompts and all five techniques are the same two moves: **write a better search input**, then
**run the unchanged retrieve-and-generate chain**. Multi-Query, RAG-Fusion and Step-Back write alternative
questions; Decomposition writes sub-questions and needs a second generation pass to combine them; HyDE writes a
passage instead of a question.

That is also the honest summary of the cost. The baseline asks the model **once** per question. Multi-Query,
RAG-Fusion and Step-Back ask it **twice**; HyDE and Decomposition **twice or more**, and Decomposition
multiplies that by the number of sub-questions. None of it is free, and the payoff depends entirely on whether
your question was actually the problem.

Each function below is the complete strategy, built only from the helpers in the setup section.

In [66]:
def rag_multi_query(question):
    """N paraphrases -> retrieve each -> unique union -> answer."""
    variants = generate_queries(question, MULTI_QUERY_PROMPT, CONFIG['num_query_variants'])
    union = unique_union([retrieve(q) for q in variants])
    union.sort(key=lambda hit: hit['score'], reverse=True)
    return answer_question(question, context_from(union))


def rag_fusion(question):
    """N paraphrases -> retrieve each -> fuse by rank -> answer."""
    variants = generate_queries(question, FUSION_PROMPT, CONFIG['num_fusion_queries'])
    fused = reciprocal_rank_fusion([retrieve(q) for q in variants])
    return answer_question(question, context_from([hit for hit, _ in fused]))


def rag_decomposition(question):
    """Sub-questions -> answer each in isolation -> synthesize over their evidence."""
    subs = generate_queries(question, DECOMPOSITION_PROMPT, CONFIG['num_sub_questions'])
    steps = individual_answers(question, subs)
    evidence = unique_union([retrieve(q) for q in subs])
    return synthesize(question, steps, evidence=context_from(evidence))


def rag_step_back(question):
    """Original question + a step-back question -> two labelled contexts -> answer."""
    step_q = step_back_question(question)
    context = (
        '# Context retrieved with the original question:\n'
        f'{context_from(retrieve(question))}\n\n'
        f'# Context retrieved with the step-back question ({step_q}):\n'
        f'{context_from(retrieve(step_q))}'
    )
    return answer_question(question, context)


def rag_hyde(question):
    """A hypothetical passage -> retrieve with the passage -> answer the original question."""
    passage = llm(HYDE_PROMPT.format(question=question), max_tokens=400)
    return answer_question(question, context_from(retrieve(passage)))


RAG_VARIANTS = {
    'baseline': baseline_rag,
    'multi-query': rag_multi_query,
    'rag-fusion': rag_fusion,
    'decomposition': rag_decomposition,
    'step-back': rag_step_back,
    'hyde': rag_hyde,
}

for name in RAG_VARIANTS:
    print(f'{name:<15} {RAG_VARIANTS[name].__doc__.splitlines()[0]}')

baseline        The Phase 1 pipeline: one query, its top-k chunks, one answer.
multi-query     N paraphrases -> retrieve each -> unique union -> answer.
rag-fusion      N paraphrases -> retrieve each -> fuse by rank -> answer.
decomposition   Sub-questions -> answer each in isolation -> synthesize over their evidence.
step-back       Original question + a step-back question -> two labelled contexts -> answer.
hyde            A hypothetical passage -> retrieve with the passage -> answer the original question.


The second question is deliberately **not** answerable from this article. It is the control: every strategy
should decline it rather than invent an answer, and any that does not has drifted away from its own retrieved
context.

Read the declines closely rather than just counting them. "The context does not contain the answer" is the
response you want — it is the model citing its own evidence. "I'm sorry, but I don't have any information
about X" sounds equally cautious but is the model falling back on what it thinks it knows, which is a weaker
signal: it would say the same thing if the corpus were empty.

This is the slowest cell in the notebook — six strategies, two questions, each with its own generation calls.
Shorten `QUESTIONS` to a single question if you only want to skim it.

In [67]:
QUESTIONS = [
    'What is Task Decomposition?',
    'Who is Msawenkosi Magwaza?',
]

for q in QUESTIONS:
    print('=' * 78)
    print('Q:', q)
    for name, strategy in RAG_VARIANTS.items():
        print('-' * 78)
        print(f'[{name}]')
        print(strategy(q))
    print()

Q: What is Task Decomposition?
------------------------------------------------------------------------------
[baseline]
Task Decomposition is the process of breaking a complex task into smaller, manageable steps or sub‑goals so that an agent can plan and execute them more easily. It is often achieved through prompting techniques such as Chain of Thought (CoT) or Tree of Thoughts, and can be performed by an LLM with simple prompts, by using task‑specific instructions, or with human input. [1]
------------------------------------------------------------------------------
[multi-query]
Task Decomposition is the process of breaking a complicated task into many smaller, manageable steps that an agent can plan and execute. It can be performed by an LLM with simple prompting, by using task‑specific instructions, or with human inputs, and is exemplified by techniques such as Chain of Thought and Tree of Thoughts. [1]
----------------------------------------------------------------------------

## What to take away

**When query transformation helps**

- The question is phrased far from how the document is written — Multi-Query and RAG-Fusion.
- The question is really several questions — Decomposition.
- The question is too specific to match a passage that contains it — Step-Back.
- The vocabulary simply does not overlap — HyDE.

**When it does not**

- Retrieval was already fine, in which case every strategy has only added latency and cost.
- The answer is not in the corpus. Every one of these techniques is a *retrieval* improvement: none of them can
  manufacture a document that was never indexed.
- You are offline, or latency matters more than coverage. A rewrite round-trip is a real cost, and the baseline
  is a strong default.

**One more thing worth keeping.** The store was not re-embedded, the prompt template was not improved, and the
model never changed. Every difference you saw came from changing the string that got embedded. If your retrieval
is weak, that string is a good place to look before reaching for a bigger model.